# VisionGuard — Phase B: training + tracking (Tasks 2, 3) — 23L-0764
Runtime → **T4 GPU**. Same secrets as Phase A. Run all (≈45–60 min). Downloads `phaseB_results.zip` at the end.

In [ ]:
# --- setup: same in both notebooks (Colab's own TensorFlow/Keras pair; pinning another Keras breaks fit() on TF 2.21) -------------------------------------------------
from google.colab import userdata
import os, subprocess, datetime
GH_USER = 'Anasnaveed69'
REPO = '23L-0764_A2_Ops'
U, T = userdata.get('DAGSHUB_USER'), userdata.get('DAGSHUB_TOKEN')   # Colab Secrets panel (key icon)
os.chdir('/content')
!rm -rf {REPO} && git clone -q https://github.com/{GH_USER}/{REPO}.git
os.chdir(f'/content/{REPO}')
!pip -q install "mlflow==2.22.5" keras-tuner==1.4.7 "dvc[s3]==3.67.1" python-dotenv -r requirements-serve.txt
!pip -q install -e . --no-deps
!git config user.name "Anas Naveed Butt" && git config user.email "{U}@users.noreply.dagshub.com"
# MLflow -> DagsHub, read from .env (never committed)
open('.env','w').write(f'''MLFLOW_TRACKING_URI=https://dagshub.com/{U}/{REPO}.mlflow
MLFLOW_TRACKING_USERNAME={U}
MLFLOW_TRACKING_PASSWORD={T}
''')
print(datetime.datetime.now().strftime('%Y-%m-%d %H:%M'), '| DagsHub user:', U, '| repo:', REPO)
!git log --oneline -3

In [ ]:
os.environ['VG_CACHE_DIR'] = '/content/tfcache'   # decoded images cached on disk across runs
!dvc remote modify origin --local access_key_id {T} && dvc remote modify origin --local secret_access_key {T}
!dvc pull -q && find data/raw -type f | wc -l && wc -l data/splits/*.csv
import tensorflow as tf; print(tf.__version__, tf.config.list_physical_devices('GPU'))

## 3.1 MLflow toy — local

In [ ]:
!python labs/mlflow_toy.py --lr 0.001 --local
!python labs/mlflow_toy.py --lr 0.01 --local
!nohup mlflow ui --backend-store-uri ./mlruns_toy --port 5000 > /tmp/mlflow_ui.log 2>&1 &
import time; time.sleep(6)
from google.colab import output; output.serve_kernel_port_as_window(5000, anchor_text='Open local MLflow UI')

## 3.2 Same toy run → DagsHub

In [ ]:
!python labs/mlflow_toy.py --lr 0.001

## 2.7 toy tuner · 2.1 augmented batch

In [ ]:
!python labs/tuner_toy.py 2>&1 | grep -v "^Epoch\|step -" | tail -30
!python scripts/04_show_batch.py

## 2.2–2.5, 3.3–3.5: all experiment runs (8 runs)

In [ ]:
!python scripts/run_experiments.py 2>&1 | grep -v ' step - \|^Epoch ' 

## 2.7 Keras Tuner on VisionGuard

In [ ]:
!python scripts/tune.py 2>&1 | grep -v ' step - \|^Epoch ' | tail -40

## 3.5 compare · 3.6 registry · export · 2.6 errors + Grad-CAM

In [ ]:
!python scripts/compare_runs.py
!python scripts/register_model.py
!python scripts/export_champion.py
import json; best = json.load(open('report/generated/best_run.json')); print(best)
!python scripts/error_analysis.py --run-dir outputs/{best['run']}

In [ ]:
# 3.6: load by alias + classify one test image; 4.1: inference module on two known images
import pandas as pd
t = pd.read_csv('data/splits/test.csv')
ai, real = t[t.label==1].path.iloc[0], t[t.label==0].path.iloc[0]
!python scripts/load_champion.py {ai}
!python scripts/infer_cli.py {ai} {real}

## Download results

In [ ]:
!zip -qr /content/phaseB_results.zip report/generated report/figures/auto models/champion.keras models/champion.json outputs/run_comparison.csv outputs/tuner
from google.colab import files; files.download('/content/phaseB_results.zip')